In [ ]:
EXPECTED={'complete': True, 'model': 'Octen/Octen-Embedding-8B', 'dim': 4096, 'chunks': 270405, 'parts': 3, 'new_model_forward_inputs': 0, 'vectors_bitwise_preserved': True, 'all_ids_match_corpus_part_order': True, 'files': {'embed_manifest.json': {'bytes': 2919, 'sha256': '35e10db14ae6203ffcea59734e67b15474b82e639a8527907ece642a8a41048c'}, 'vectors/en-part-001.npy': {'bytes': 114816, 'sha256': 'bce42387976215e9fac3e11febc4f0fd9104ca543747be2b2e3f48064f733c53'}, 'vectors/vi-part-007-ids.parquet': {'bytes': 526152, 'sha256': 'fc9d06267e6eb98c65a73f35f174bcd8ba75172c4480ced025e089d6b368eed0'}, 'vectors/zh-part-001-ids.parquet': {'bytes': 1294525, 'sha256': '04fe3a8058bf82576af8f237535664f4b6d5c8ef308ab2062bcfd28010b6c084'}, 'vectors/zh-part-001.npy': {'bytes': 1429119104, 'sha256': 'cbf17f05ddd92de1d1abf482757765525f9a5be7dddd79cc55fcc04da97b0beb'}, 'vectors/en-part-001-ids.parquet': {'bytes': 692, 'sha256': '8c7ac9224183e872872151a3f8f2de31c433109c24b74739ecdbd998ee3bac25'}, 'vectors/vi-part-007.npy': {'bytes': 785924224, 'sha256': '7cac683ca34cb57e0fdd86990fb2b732eca486a193cae39edcd98e43229f92d9'}}}
DATASET='ntklinhfitus/medxir-vectors-octen-8b-additions'
from pathlib import Path
import hashlib,json
import numpy as np
import pyarrow.parquet as pq
roots=list(Path('/kaggle/input').rglob('embed_manifest.json'))
corpora=list(Path('/kaggle/input').rglob('chunk_manifest.json'))
assert len(roots)==len(corpora)==1
root=roots[0].parent; corpus=corpora[0].parent
actual={p.relative_to(root).as_posix():p for p in root.rglob('*') if p.is_file()}
assert set(actual)==set(EXPECTED['files'])
for name,meta in EXPECTED['files'].items():
    with actual[name].open('rb') as f: digest=hashlib.file_digest(f,'sha256').hexdigest()
    assert actual[name].stat().st_size==meta['bytes'] and digest==meta['sha256'],name
m=json.loads(roots[0].read_text())
with corpora[0].open('rb') as f: corpus_sha=hashlib.file_digest(f,'sha256').hexdigest()
assert corpus_sha==m['corpus_manifest_sha256']
assert m['complete'] and m['chunks']==270405 and m['dim']==4096 and m['model']==EXPECTED['model']
assert m['new_model_forward_inputs']==0 and m['vectors_bitwise_preserved']
assert {'Qwen/Qwen3-Embedding-8B':'','Octen/Octen-Embedding-8B':' '}[m['model']]==m['doc_prefix']
counts={}; all_ids=set()
for lang,part in [('vi','007'),('en','001'),('zh','001')]:
    key=f'{lang}-part-{part}'
    ids=pq.read_table(root/f'vectors/{key}-ids.parquet',columns=['chunk_id'])['chunk_id'].to_pylist()
    expected_ids=pq.read_table(corpus/f'chunks/{lang}/part-{part}.parquet',columns=['chunk_id'])['chunk_id'].to_pylist()
    assert ids==expected_ids and len(set(ids))==len(ids) and not all_ids.intersection(ids)
    all_ids.update(ids)
    v=np.load(root/f'vectors/{key}.npy',mmap_mode='r',allow_pickle=False)
    assert v.dtype==np.float16 and v.shape==(len(ids),4096)
    for start in range(0,len(v),8192):
        block=np.array(v[start:start+8192],dtype=np.float32)
        assert np.isfinite(block).all() and np.max(abs(np.linalg.norm(block,axis=1)-1))<.01
    counts[lang]=len(ids); del v
assert counts==dict(vi=95938,en=14,zh=174453) and len(all_ids)==270405
report=dict(complete=True,dataset=DATASET,version=1,model=m['model'],dim=4096,
    rows=len(all_ids),rows_by_language=counts,all_file_checksums_verified=True,
    chunk_ids_and_order_match=True,all_vectors_finite_and_unit_norm=True,
    vectors_bitwise_unchanged_from_verified_source=True,baseline_modified=False)
Path('/kaggle/working/published_vectors_validation.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
